In [1]:
# ============================================
# STEP 1: IMPORT REQUIRED LIBRARIES
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# NLP libraries
import nltk
import string

In [3]:
# ============================================
# LOAD THE IMDB DATASET
# ============================================

df = pd.read_csv("IMDB Dataset.csv")

# Display first 5 rows
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
# Check number of rows and columns
print("Dataset shape:", df.shape)

# Check column names
print("\nColumns:")
print(df.columns)

# Check data types and dataset information
print("\nDataset Information:")
df.info()

Dataset shape: (50000, 2)

Columns:
Index(['review', 'sentiment'], dtype='object')

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [7]:
# ============================================
# STEP 2: BASIC DATA QUALITY CHECKS
# ============================================

# Check missing values
print("Missing Values:")
print(df.isnull().sum())

# Check duplicate rows
print("\nDuplicate Rows:")
print(df.duplicated().sum())

# Check class distribution
print("\nSentiment Distribution:")
print(df["sentiment"].value_counts())

Missing Values:
review       0
sentiment    0
dtype: int64

Duplicate Rows:
418

Sentiment Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [9]:
# ============================================
# STEP 3: REMOVE DUPLICATE REVIEWS
# ============================================

# Remove duplicate rows
df.drop_duplicates(inplace=True)

# Reset index after removing duplicates
df.reset_index(drop=True, inplace=True)

# Verify duplicates are removed
print("Duplicate Rows:", df.duplicated().sum())

# Check new dataset shape
print("Dataset Shape:", df.shape)

# Check class distribution again
print("\nSentiment Distribution:")
print(df["sentiment"].value_counts())

Duplicate Rows: 0
Dataset Shape: (49582, 2)

Sentiment Distribution:
sentiment
positive    24884
negative    24698
Name: count, dtype: int64


### Data Inspection Conclusion

- The dataset contains no missing values.
- 418 duplicate records were identified and removed.
- The target variable is approximately balanced between positive and negative sentiments.
- Therefore, no missing-value treatment or class-balancing technique is required.

In [13]:
# ============================================
# STEP 4: TEXT PREPROCESSING
# ============================================

import nltk

# Required for tokenization
nltk.download("punkt")
nltk.download("punkt_tab")

# Required for stopwords
nltk.download("stopwords")

# Required for lemmatization
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to /Users/likithcp/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/likithcp/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/likithcp/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/likithcp/nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/likithcp/nltk_data...


True

In [15]:
# Import NLP preprocessing tools

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

import string

# English stopwords
stop_words = set(stopwords.words("english"))

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

In [21]:
# Import regular expressions for removing HTML tags
import re

def preprocess_text(text):

    # 1. Convert text to lowercase
    text = text.lower()

    # 2. Remove HTML tags such as <br />, <p>, etc.
    text = re.sub(r"<.*?>", " ", text)

    # 3. Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # 4. Tokenize text into individual words
    tokens = word_tokenize(text)

    # 5. Remove stopwords
    tokens = [
        word for word in tokens
        if word not in stop_words
    ]

    # 6. Lemmatize words
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    # 7. Join tokens back into cleaned text
    return " ".join(tokens)

In [23]:
# Test preprocessing on one review

print("ORIGINAL REVIEW:")
print(df["review"].iloc[0])

print("\nCLEANED REVIEW:")
print(preprocess_text(df["review"].iloc[0]))

ORIGINAL REVIEW:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show 

In [29]:
print(preprocess_text(df["review"].iloc[0]))

one reviewer mentioned watching 1 oz episode youll hooked right exactly happened first thing struck oz brutality unflinching scene violence set right word go trust show faint hearted timid show pull punch regard drug sex violence hardcore classic use word called oz nickname given oswald maximum security state penitentary focus mainly emerald city experimental section prison cell glass front face inwards privacy high agenda em city home manyaryans muslim gangsta latino christian italian irish moreso scuffle death stare dodgy dealing shady agreement never far away would say main appeal show due fact go show wouldnt dare forget pretty picture painted mainstream audience forget charm forget romanceoz doesnt mess around first episode ever saw struck nasty surreal couldnt say ready watched developed taste oz got accustomed high level graphic violence violence injustice crooked guard wholl sold nickel inmate wholl kill order get away well mannered middle class inmate turned prison bitch due l

In [31]:
# ============================================
# APPLY PREPROCESSING TO ENTIRE DATASET
# ============================================

df["cleaned_review"] = df["review"].apply(preprocess_text)

df[["review", "cleaned_review", "sentiment"]].head()

,review,cleaned_review,sentiment
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching 1 oz episode y...,positive
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...,positive
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...,positive
3,Basically there's a family where a little boy ...,basically there family little boy jake think t...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...,positive


In [35]:
# Compare original and cleaned review

print("ORIGINAL:")
print(df["review"].iloc[0])

print("\nCLEANED:")
print(df["cleaned_review"].iloc[0])

ORIGINAL:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due 

In [37]:
# ============================================
# STEP 6: DEFINE FEATURES AND TARGET
# ============================================

# X contains the preprocessed movie reviews
X = df["cleaned_review"]

# y contains the target labels: positive / negative
y = df["sentiment"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (49582,)
y shape: (49582,)


In [39]:
# ============================================
# STEP 7: TRAIN-TEST SPLIT
# ============================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,       # 20% testing data
    random_state=42,      # Reproducible split
    stratify=y            # Maintain positive/negative ratio
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (39665,)
X_test : (9917,)
y_train: (39665,)
y_test : (9917,)


In [41]:
# ============================================
# STEP 8: TF-IDF FEATURE EXTRACTION
# ============================================

from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=5000
)

# Learn vocabulary from training data and transform it
X_train_tfidf = tfidf.fit_transform(X_train)

# Only transform test data using learned vocabulary
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF Shape:", X_train_tfidf.shape)
print("Testing TF-IDF Shape :", X_test_tfidf.shape)

Training TF-IDF Shape: (39665, 5000)
Testing TF-IDF Shape : (9917, 5000)


In [43]:
from sklearn.feature_extraction.text import CountVectorizer

# Bag-of-Words
bow = CountVectorizer(max_features=5000)

# Fit only on training data
X_train_bow = bow.fit_transform(X_train)

# Transform test data
X_test_bow = bow.transform(X_test)

print("BoW Train Shape:", X_train_bow.shape)
print("BoW Test Shape :", X_test_bow.shape)

BoW Train Shape: (39665, 5000)
BoW Test Shape : (9917, 5000)


In [45]:
# ============================================
# STEP 9: TRAIN LOGISTIC REGRESSION MODEL
# ============================================

from sklearn.linear_model import LogisticRegression

# Create Logistic Regression model
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train the model using TF-IDF features
model.fit(X_train_tfidf, y_train)

print("Model training completed.")

Model training completed.


In [47]:
# ============================================
# STEP 10: MAKE PREDICTIONS
# ============================================

y_pred = model.predict(X_test_tfidf)

print("Predictions completed.")

Predictions completed.


In [49]:
# ============================================
# STEP 11: MODEL EVALUATION
# ============================================

from sklearn.metrics import accuracy_score, f1_score, classification_report

accuracy = accuracy_score(y_test, y_pred)

# Since labels are strings, specify positive class
f1 = f1_score(
    y_test,
    y_pred,
    pos_label="positive"
)

print("Accuracy:", accuracy)
print("F1 Score:", f1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.888978521730362
F1 Score: 0.8905675380180896

Classification Report:
              precision    recall  f1-score   support

    negative       0.90      0.88      0.89      4940
    positive       0.88      0.90      0.89      4977

    accuracy                           0.89      9917
   macro avg       0.89      0.89      0.89      9917
weighted avg       0.89      0.89      0.89      9917



In [51]:

from sklearn.linear_model import LogisticRegression

bow_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train using BoW features
bow_model.fit(X_train_bow, y_train)

# Make predictions
y_pred_bow = bow_model.predict(X_test_bow)





In [57]:
# ============================================
# EVALUATE BAG-OF-WORDS MODEL
# ============================================

from sklearn.metrics import accuracy_score, f1_score, classification_report

bow_accuracy = accuracy_score(y_test, y_pred_bow)

bow_f1 = f1_score(
    y_test,
    y_pred_bow,
    pos_label="positive"
)

print("BoW Accuracy:", bow_accuracy)
print("BoW F1 Score:", bow_f1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_bow))

BoW Accuracy: 0.875567207824947
BoW F1 Score: 0.875930022119445

Classification Report:
              precision    recall  f1-score   support

    negative       0.87      0.88      0.88      4940
    positive       0.88      0.88      0.88      4977

    accuracy                           0.88      9917
   macro avg       0.88      0.88      0.88      9917
weighted avg       0.88      0.88      0.88      9917



In [59]:
# ============================================
# STEP 15: TOP 10 IMPORTANT WORDS PER CLASS
# ============================================

import numpy as np

# Get all feature/word names learned by TF-IDF
feature_names = tfidf.get_feature_names_out()

# Get Logistic Regression coefficients
coefficients = model.coef_[0]

# Highest positive coefficients = important positive words
top_positive_indices = np.argsort(coefficients)[-10:][::-1]

# Lowest/most negative coefficients = important negative words
top_negative_indices = np.argsort(coefficients)[:10]

# Get actual words
top_positive_words = feature_names[top_positive_indices]
top_negative_words = feature_names[top_negative_indices]

print("Top 10 Important Words for POSITIVE Sentiment:")
print(top_positive_words)

print("\nTop 10 Important Words for NEGATIVE Sentiment:")
print(top_negative_words)

Top 10 Important Words for POSITIVE Sentiment:
['great' 'excellent' '710' 'perfect' 'best' 'amazing' 'favorite'
 'wonderful' 'loved' 'brilliant']

Top 10 Important Words for NEGATIVE Sentiment:
['worst' 'awful' 'waste' 'bad' 'boring' 'poor' 'terrible' 'nothing'
 'horrible' 'worse']
